In [25]:
# Paste in Jupyter, paste output back
from pymongo import MongoClient
import os

uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
client = MongoClient(uri)
db = client[os.environ.get("MONGO_DATABASE", "quants_lab")]

pipeline = [
    {"$match": {"trading_pair": "XMR-USDT", "connector": {"$in": ["mexc", "nonkyc"]}}},
    {"$group": {
        "_id": {"connector": "$connector", "interval": "$interval"},
        "count": {"$sum": 1},
        "first_ts": {"$min": "$timestamp"},
        "last_ts":  {"$max": "$timestamp"},
    }},
    {"$sort": {"_id.connector": 1, "_id.interval": 1}},
]
for doc in db["candles"].aggregate(pipeline):
    c = doc["_id"]["connector"]
    i = doc["_id"]["interval"]
    n = doc["count"]
    span_days = (doc["last_ts"] - doc["first_ts"]) / 86400
    print(f"{c:>8}  {i:>4}  {n:>7} bars   {span_days:6.1f} days")

    mexc   15m    34599 bars    360.4 days
    mexc    1d     2318 bars   2317.0 days
    mexc    1h     5410 bars    225.4 days
    mexc    1m   108735 bars     75.5 days
    mexc    4h     1351 bars    225.0 days
    mexc    5m   103798 bars    360.4 days
    mexc    8h      540 bars    179.7 days
  nonkyc   12h      360 bars    179.5 days
  nonkyc   15m    96889 bars   1053.5 days
  nonkyc    1d     1079 bars   1079.0 days
  nonkyc    1h    25541 bars   1079.5 days
  nonkyc    4h     6446 bars   1079.3 days
  nonkyc    5m   201274 bars    730.5 days
  nonkyc    8h      540 bars    179.7 days
